# NBV Agent Benchmark Analysis

This notebook analyzes the raw benchmark data generated by the `benchmark.py` script.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Set premium plotting style
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({'font.size': 12, 'figure.dpi': 150})

## 1. Load and Preprocess Data

We load the raw CSV, convert coverage to a percentage, and calculate the cumulative fuel consumed at each step.

In [ ]:
data_path = "artefacts/benchmark/benchmark_raw_data.csv"
try:
    df = pd.read_csv(data_path)
    print(f"Data loaded successfully: {len(df)} rows")
except FileNotFoundError:
    print(f"File not found at {data_path}. Please run benchmark.py first.")

# Calculate Fuel Consumed
initial_fuel = 50.0  # fuel_budget from config
df['fuel_consumed'] = initial_fuel - df['fuel_remaining']

# Convert coverage to percentage
df['coverage_pct'] = df['coverage'] * 100.0

## 2. Visualization Helper

This function plots the average metric over time steps and overlays a shaded region representing the absolute minimum and maximum values achieved across all objects at that specific step.

In [ ]:
def plot_minmax_band(df, metric, title, ylabel, split_name):
    subset = df[df['dataset_split'] == split_name]
    if subset.empty:
        print(f"No data available for split: {split_name}")
        return
        
    plt.figure(figsize=(10, 6))
    
    policies = subset['policy'].unique()
    colors = sns.color_palette("Set1", len(policies))
    
    for i, policy in enumerate(policies):
        policy_data = subset[subset['policy'] == policy]
        
        # Aggregate by time step
        step_stats = policy_data.groupby('step')[metric].agg(['mean', 'min', 'max']).reset_index()
        
        # Plot mean line
        plt.plot(step_stats['step'], step_stats['mean'], 
                 label=f"{policy} (Average)", color=colors[i], linewidth=2.5)
        
        # Plot Min-Max shaded band
        plt.fill_between(
            step_stats['step'], 
            step_stats['min'], 
            step_stats['max'], 
            color=colors[i], 
            alpha=0.15, 
            label=f"{policy} (Min/Max Spread)"
        )
        
    plt.title(f"{title} ({split_name} Set)", fontsize=15, fontweight='bold', pad=15)
    plt.xlabel("Time Step", fontsize=13)
    plt.ylabel(ylabel, fontsize=13)
    
    # Condense legend
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    plt.legend(by_label.values(), by_label.keys(), loc='center left', bbox_to_anchor=(1.02, 0.5))
    
    plt.tight_layout()
    plt.show()

## 3. Test Set Performance

The Test set represents unseen objects and is the primary indicator of your agent's generalizability. 

In [ ]:
plot_minmax_band(df, metric='coverage_pct', 
                 title='Percentage Inspected Over Time', 
                 ylabel='Surface Coverage (%)', 
                 split_name='Test')

plot_minmax_band(df, metric='fuel_consumed', 
                 title='Cumulative Fuel Consumption Over Time', 
                 ylabel='Fuel Consumed (Δv)', 
                 split_name='Test')

## 4. Train Set Performance

Visualizing the train set helps diagnose if the agent is overfitting (i.e., performing significantly better here than on the test set).

In [ ]:
plot_minmax_band(df, metric='coverage_pct', 
                 title='Percentage Inspected Over Time', 
                 ylabel='Surface Coverage (%)', 
                 split_name='Train')

plot_minmax_band(df, metric='fuel_consumed', 
                 title='Cumulative Fuel Consumption Over Time', 
                 ylabel='Fuel Consumed (Δv)', 
                 split_name='Train')

## 5. Final Summary Statistics

This table extracts the performance at the very last step of each episode, providing a clean summary of final coverage and total fuel costs.

In [ ]:
# Identify the final step for each unique episode
idx = df.groupby(['dataset_split', 'policy', 'model_name', 'loop_id'])['step'].idxmax()
final_steps = df.loc[idx]

# Calculate Mean and Std Deviation
summary = final_steps.groupby(['dataset_split', 'policy'])[['coverage_pct', 'fuel_consumed']].agg(['mean', 'std']).round(2)
summary.columns = ['Avg Final Coverage (%)', 'Coverage Std', 'Avg Total Fuel', 'Fuel Std']

print("--- Final Performance Summary ---")
display(summary)